# Course 5: Winning Online AI Competitions
## Session 4: Leakage

**Goal:** plant three kinds of leak, measure how much each inflates the validation score, and learn the checks that find them.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on synthetic or built-in data. Cells labelled NOT RUN need a Kaggle account, a competition's data, a GPU or an extra library.

In [ ]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
def make_tabular(n=6000, seed=0):
    """A synthetic binary task with numeric columns, one high-cardinality category, groups and a date."""
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({"id": np.arange(n)})
    for j in range(6): df[f"num{j}"] = rng.normal(size=n)
    n_shops = 300
    shop_effect = rng.normal(scale=1.0, size=n_shops)
    df["shop"] = rng.zipf(1.5, n).clip(max=n_shops) - 1                         # a few big shops, many small ones
    df["customer"] = rng.integers(0, n // 8, n)                                 # about 8 rows per customer
    cust_effect = rng.normal(scale=1.2, size=n // 8)
    df["day"] = np.sort(rng.integers(0, 365, n))
    logit = 0.8 * df.num0 - 0.6 * df.num1 + 0.5 * df.num2 * df.num3 + shop_effect[df["shop"]] + cust_effect[df.customer] + 0.004 * df.day - 1.0
    df["target"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    return df
NUM = [f"num{j}" for j in range(6)]

### 1. An honest baseline

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import roc_auc_score
df = make_tabular(6000); rng = np.random.default_rng(0)
train, test = df.iloc[:4500].reset_index(drop=True), df.iloc[4500:].reset_index(drop=True)
cv = KFold(5, shuffle=True, random_state=0)
def cv_and_test(feats, tr=train, te=test, model=None):
    model = model or HistGradientBoostingClassifier(max_iter=100, random_state=0)
    oof = cross_val_predict(model, tr[feats], tr.target, cv=cv, method="predict_proba")[:, 1]
    return roc_auc_score(tr.target, oof), roc_auc_score(te.target, model.fit(tr[feats], tr.target).predict_proba(te[feats])[:, 1])
base = cv_and_test(NUM); print("honest model: CV AUC %.3f | test AUC %.3f" % base)

### 2. Leak one: a column that knows the answer
A column recorded after the outcome (for example 'refund issued' when predicting fraud). It exists in the training file and is empty or meaningless at prediction time.

In [ ]:
train1, test1 = train.copy(), test.copy()
train1["follow_up_calls"] = train1.target * rng.poisson(2, len(train1)) + rng.poisson(0.2, len(train1))      # filled in after the event
test1["follow_up_calls"] = rng.poisson(0.2, len(test1))                                                       # not known yet at prediction time
leak1 = cv_and_test(NUM + ["follow_up_calls"], train1, test1)
print("with the leaky column: CV AUC %.3f | test AUC %.3f" % leak1)
assert leak1[0] > base[0] + 0.05 and leak1[1] < leak1[0] - 0.05

How to find it: a single feature with a very high AUC by itself, a feature importance that dwarfs the rest, or a column whose distribution differs between train and test.

In [ ]:
single = {c: max(a := roc_auc_score(train1.target, train1[c]), 1 - a) for c in NUM + ["follow_up_calls"]}
print("AUC of each feature alone:", {k: round(v, 3) for k, v in sorted(single.items(), key=lambda kv: -kv[1])})
print("mean of follow_up_calls in train %.2f | in test %.2f" % (train1.follow_up_calls.mean(), test1.follow_up_calls.mean()))

### 3. Leak two: preprocessing that saw the validation rows
Selecting features, or fitting any statistic that uses the target, on all rows before cross-validation. Here: 2,000 pure-noise columns, 'keep the 20 most correlated with the target', then cross-validate.

In [ ]:
n = 300; noise = pd.DataFrame(rng.normal(size=(n, 2000))); y_noise = pd.Series(rng.integers(0, 2, n))        # there is nothing to learn here
top = noise.corrwith(y_noise).abs().sort_values(ascending=False).index[:20]                                   # chosen using ALL rows: the leak
lr = LogisticRegression(max_iter=1000)
leaky = roc_auc_score(y_noise, cross_val_predict(lr, noise[top], y_noise, cv=cv, method="predict_proba")[:, 1])
oof = np.zeros(n)
for tr, va in cv.split(noise):
    top_k = noise.iloc[tr].corrwith(y_noise.iloc[tr]).abs().sort_values(ascending=False).index[:20]            # chosen inside the fold
    oof[va] = lr.fit(noise.iloc[tr][top_k], y_noise.iloc[tr]).predict_proba(noise.iloc[va][top_k])[:, 1]
honest = roc_auc_score(y_noise, oof)
print(f"pure noise: selection before CV gives AUC {leaky:.3f} | selection inside each fold gives AUC {honest:.3f}")
assert leaky > 0.8 and honest < 0.65

An AUC above 0.8 on random numbers. The rule: anything that looks at the target (feature selection, target encoding, choosing a threshold, stacking) must happen inside the fold. Session 5 applies it to target encoding.

### 4. Leak three: the same entity on both sides
Duplicate or near-duplicate rows, or rows from the same patient, appear in both training and validation. Tree models memorize them.

In [ ]:
train3 = pd.concat([train, train.sample(1500, random_state=0)], ignore_index=True)                           # 1,500 rows entered twice
deep = HistGradientBoostingClassifier(max_iter=300, max_depth=None, min_samples_leaf=1, learning_rate=0.2, random_state=0)
leak3 = cv_and_test(NUM, train3, test, deep); clean3 = cv_and_test(NUM, train, test, deep)
print("no duplicates  : CV AUC %.3f | test AUC %.3f" % clean3)
print("with duplicates: CV AUC %.3f | test AUC %.3f" % leak3)
dup = train3.duplicated(subset=NUM, keep=False).mean()
print(f"check: share of training rows that have an exact duplicate on the feature columns: {dup:.2f}")
assert leak3[0] > clean3[0] + 0.02 and abs(leak3[1] - clean3[1]) < 0.03

### 5. A leak the organisers made: row order
Sometimes the file itself carries information, for example when rows were sorted by target before ids were assigned. Always check whether the id predicts the target.

In [ ]:
train5 = train.sort_values("target", kind="stable").reset_index(drop=True); train5["id"] = np.arange(len(train5))
a = roc_auc_score(train5.target, train5.id); print("AUC of the id column alone: %.3f in the sorted file | %.3f in the original" % (a, roc_auc_score(train.target, train.id)))

If you find a leak like this in a real competition, read the rules and the forum. Using it is often allowed, sometimes the competition is restarted, and a model built on it is useless outside the competition either way.

### 6. A leak checklist
1. Does any single feature score far above the rest alone?
2. Is every fitted step (scaler, encoder, selector, imputer using the target) inside the fold? Use a `Pipeline`.
3. Are there duplicates, or groups, shared across folds or between train and test?
4. Does the id, the row order or a timestamp predict the target?
5. Is the CV score much better than the leaderboard? Something leaks.
6. Would this feature exist at the moment a real prediction is made?

### TODO
1. Rewrite leak two with `sklearn.pipeline.Pipeline` and `SelectKBest` so that `cross_val_predict` does the right thing by construction.
2. Plant a time leak: a feature equal to the mean target of the next 10 rows. Detect it.
3. Search the forum of a finished competition for the word 'leak'. Summarize what was found in five sentences.